In [ ]:
import numpy as np
import soundfile

from audio_preprocessing import (
    load_audio_file,
    resample_audio_to_target_sr,
    normalize_audio,
    segment_audio,
    compute_mel_spectrogram,
    extract_f0,
)
from drawings import waveform_chart, comparison_chart, spectrogram_chart, f0_chart
from config import AudioConfig

In [ ]:
cfg = AudioConfig()

# Load Audio

In [ ]:
audio, orig_sr = load_audio_file("PATH_TO_YOUR_AUDIO")
print(f"loaded: {len(audio)} samples @ {orig_sr} Hz ({len(audio) / orig_sr:.1f}s)")

In [ ]:
waveform_chart(audio, int(orig_sr), f"After load — {orig_sr} Hz", color="#6366f1")

# Resample Audio (to 22050 Hz)

In [ ]:
audio_loaded = audio.copy()
audio = resample_audio_to_target_sr(audio, orig_sr)
print(f"resampled: {len(audio)} samples @ {cfg.target_sr} Hz")

In [ ]:
comparison_chart(
    [
        (audio_loaded, orig_sr, f"original · {orig_sr} Hz"),
        (audio, cfg.target_sr, f"resampled · {cfg.target_sr} Hz"),
    ]
)

# Normalise Audio

In [ ]:
audio_pre_norm = audio.copy()
audio = normalize_audio(audio)
print(
    f"normalized: peak {np.abs(audio).max():.3f} (was {np.abs(audio_pre_norm).max():.3f})"
)

In [ ]:
comparison_chart(
    [
        (audio_pre_norm, cfg.target_sr, "pre-normalize"),
        (audio, cfg.target_sr, "normalized · -23 LUFS"),
    ]
)

# Segment Audio

In [ ]:
segments = segment_audio(audio)
first_segement = segments.__next__()
waveform_chart(first_segement, cfg.target_sr, "The first segment", color="#6366f1")

## Example Spectrogram

In [ ]:
mel = compute_mel_spectrogram(first_segement)
spectrogram_chart(mel, cfg.target_sr, title="The spectrogram of the first segment")

# Extract F0 Contours using CREPE

In [ ]:
time, frequency, confidence = extract_f0(first_segement)
f0_chart(time, frequency, confidence=confidence)

# Save it to example.wav in the project's root

In [ ]:
soundfile.write("example.wav", audio, cfg.target_sr, format="wav")